# CNN-Based Advertisement Detection - Training Notebook

This notebook trains a custom CNN classifier to detect advertisements in images.

## Overview
- **Task:** Binary image classification (ad vs non-ad)
- **Model:** Custom CNN architecture from `models.py`
- **Data:** Synthetic images generated via Google Imagen API
- **Output:** TensorFlow.js model for Chrome extension deployment

## Dataset Structure
```
imageGen/output/
├── ads/          # Advertisement images
└── non_ads/      # Non-advertisement images
```

## 1. Setup and Imports

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

# TensorFlow and Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import (
    EarlyStopping, 
    ModelCheckpoint, 
    ReduceLROnPlateau,
    TensorBoard
)

# Image processing
from PIL import Image
import cv2

# Metrics and evaluation
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_curve, 
    roc_auc_score,
    precision_recall_curve
)

# Import custom models
from models import build_custom_cnn

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2. Configuration

In [ ]:
# Paths
BASE_DIR = Path('../imageGen/output')  # Directory containing ads/ and non_ads/
OUTPUT_DIR = Path('../chromeExtension/model')  # Where to save the trained model
CHECKPOINT_DIR = Path('./checkpoints')
LOG_DIR = Path('./logs')

# Create directories if they don't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Model configuration
IMG_HEIGHT = 224
IMG_WIDTH = 224
IMG_CHANNELS = 3
INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)

# Training configuration
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
VALIDATION_SPLIT = 0.2  # 20% for validation
TEST_SPLIT = 0.1        # 10% for test (from remaining data)

# Data augmentation settings
AUGMENTATION = True

print(f"Input shape: {INPUT_SHAPE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {EPOCHS}")
print(f"Data augmentation: {AUGMENTATION}")

## 3. Data Preparation

### 3.1 Explore Dataset

In [ ]:
# Check if data directory exists
if not BASE_DIR.exists():
    print(f"ERROR: Data directory not found at {BASE_DIR}")
    print("Please generate images first using: cd ../imageGen && python main.py --num-images 100")
else:
    # Count images in each category
    ads_dir = BASE_DIR / 'ads'
    non_ads_dir = BASE_DIR / 'non_ads'
    
    if ads_dir.exists() and non_ads_dir.exists():
        num_ads = len(list(ads_dir.glob('*.png')))
        num_non_ads = len(list(non_ads_dir.glob('*.png')))
        total_images = num_ads + num_non_ads
        
        print(f"Dataset found at: {BASE_DIR}")
        print(f"  - Ads: {num_ads} images")
        print(f"  - Non-ads: {num_non_ads} images")
        print(f"  - Total: {total_images} images")
        print(f"  - Balance: {num_ads/total_images*100:.1f}% ads, {num_non_ads/total_images*100:.1f}% non-ads")
        
        # Check if we have enough data
        if total_images < 50:
            print("\n⚠️ WARNING: Very small dataset. Consider generating more images for better results.")
            print("Recommended: At least 500-1000 images per class")
    else:
        print(f"ERROR: Expected subdirectories 'ads' and 'non_ads' in {BASE_DIR}")

### 3.2 Visualize Sample Images

In [ ]:
def visualize_samples(num_samples=5):
    """Display sample images from each class."""
    ads_dir = BASE_DIR / 'ads'
    non_ads_dir = BASE_DIR / 'non_ads'
    
    # Get sample images
    ad_images = list(ads_dir.glob('*.png'))[:num_samples]
    non_ad_images = list(non_ads_dir.glob('*.png'))[:num_samples]
    
    # Create figure
    fig, axes = plt.subplots(2, num_samples, figsize=(15, 6))
    fig.suptitle('Sample Images from Dataset', fontsize=16, fontweight='bold')
    
    # Display ads
    for i, img_path in enumerate(ad_images):
        img = load_img(img_path)
        axes[0, i].imshow(img)
        axes[0, i].set_title(f'Ad {i+1}', fontsize=10)
        axes[0, i].axis('off')
    
    # Display non-ads
    for i, img_path in enumerate(non_ad_images):
        img = load_img(img_path)
        axes[1, i].imshow(img)
        axes[1, i].set_title(f'Non-Ad {i+1}', fontsize=10)
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_samples(num_samples=5)

### 3.3 Prepare Data for Training

We'll use tf.keras.utils.image_dataset_from_directory for more efficient data loading.
This automatically handles train/val splits and batching.

In [ ]:
# Create train and validation datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    BASE_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="training",
    seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='binary'  # Binary classification
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    BASE_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="validation",
    seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

# Get class names
class_names = train_ds.class_names
print(f"Class names: {class_names}")
print(f"  - Class 0: {class_names[0]}")
print(f"  - Class 1: {class_names[1]}")

# Calculate dataset sizes
train_size = tf.data.experimental.cardinality(train_ds).numpy() * BATCH_SIZE
val_size = tf.data.experimental.cardinality(val_ds).numpy() * BATCH_SIZE
print(f"\nDataset splits:")
print(f"  - Training: ~{train_size} images")
print(f"  - Validation: ~{val_size} images")

### 3.4 Data Augmentation Pipeline

In [ ]:
# Note: The custom CNN already includes rescaling (0-255 -> 0-1)
# So we don't need to add it here

if AUGMENTATION:
    # Data augmentation layers
    data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),  # ±10% rotation
        layers.RandomZoom(0.1),       # ±10% zoom
        layers.RandomTranslation(0.1, 0.1),  # ±10% shift
        layers.RandomContrast(0.2),   # Contrast adjustment
    ], name='data_augmentation')
    
    print("Data augmentation enabled:")
    print("  - Random horizontal flip")
    print("  - Random rotation (±10%)")
    print("  - Random zoom (±10%)")
    print("  - Random translation (±10%)")
    print("  - Random contrast adjustment")
else:
    data_augmentation = None
    print("Data augmentation disabled")

### 3.5 Visualize Augmented Images

In [ ]:
if AUGMENTATION:
    # Get a sample image
    for images, labels in train_ds.take(1):
        sample_image = images[0]
        sample_label = labels[0]
    
    # Apply augmentation multiple times
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    fig.suptitle('Data Augmentation Examples', fontsize=16, fontweight='bold')
    
    # Original
    axes[0, 0].imshow(sample_image.numpy().astype("uint8"))
    axes[0, 0].set_title('Original')
    axes[0, 0].axis('off')
    
    # Augmented versions
    for i in range(1, 8):
        augmented_image = data_augmentation(tf.expand_dims(sample_image, 0))
        ax = axes[i // 4, i % 4]
        ax.imshow(augmented_image[0].numpy().astype("uint8"))
        ax.set_title(f'Augmented {i}')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

### 3.6 Optimize Dataset Performance

In [ ]:
# Configure for performance
AUTOTUNE = tf.data.AUTOTUNE

# Apply augmentation to training data only
if AUGMENTATION:
    train_ds = train_ds.map(
        lambda x, y: (data_augmentation(x, training=True), y),
        num_parallel_calls=AUTOTUNE
    )

# Prefetch for performance
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("Dataset optimization complete:")
print("  - Caching enabled")
print("  - Prefetching enabled")
print("  - Parallel data loading enabled")

## 4. Build Model

Using the custom CNN architecture from `models.py`

In [ ]:
# Build the custom CNN model
model = build_custom_cnn(
    input_shape=INPUT_SHAPE,
    num_classes=1  # Binary classification
)

# Display model architecture
model.summary()

In [ ]:
# Visualize model architecture
keras.utils.plot_model(
    model,
    to_file='model_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    rankdir='TB',
    dpi=96
)

from IPython.display import Image as IPImage
IPImage('model_architecture.png')

## 5. Training Configuration

In [ ]:
# Callbacks for training
callbacks = [
    # Early stopping: stop if val_loss doesn't improve for 10 epochs
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Model checkpoint: save best model
    ModelCheckpoint(
        filepath=str(CHECKPOINT_DIR / 'best_model.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    
    # Reduce learning rate when val_loss plateaus
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    
    # TensorBoard logging
    TensorBoard(
        log_dir=str(LOG_DIR / datetime.now().strftime('%Y%m%d-%H%M%S')),
        histogram_freq=1
    )
]

print("Training callbacks configured:")
print("  - Early stopping (patience=10)")
print("  - Model checkpointing")
print("  - Learning rate reduction (factor=0.5, patience=5)")
print("  - TensorBoard logging")
print(f"\nCheckpoints will be saved to: {CHECKPOINT_DIR}")
print(f"TensorBoard logs: {LOG_DIR}")
print("\nTo view TensorBoard, run: tensorboard --logdir=./logs")

## 6. Train Model

In [ ]:
# Train the model
print("Starting training...\n")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete!")

## 7. Training History Visualization

In [ ]:
def plot_training_history(history):
    """Plot training and validation metrics."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Training History', fontsize=16, fontweight='bold')
    
    # Loss
    axes[0, 0].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[0, 0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    axes[0, 0].set_title('Model Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[0, 1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    axes[0, 1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
    axes[0, 1].set_title('Model Accuracy')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Precision
    axes[1, 0].plot(history.history['precision'], label='Train Precision', linewidth=2)
    axes[1, 0].plot(history.history['val_precision'], label='Val Precision', linewidth=2)
    axes[1, 0].set_title('Model Precision')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Recall
    axes[1, 1].plot(history.history['recall'], label='Train Recall', linewidth=2)
    axes[1, 1].plot(history.history['val_recall'], label='Val Recall', linewidth=2)
    axes[1, 1].set_title('Model Recall')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_training_history(history)

## 8. Model Evaluation

In [ ]:
# Evaluate on validation set
print("Evaluating model on validation set...\n")
val_metrics = model.evaluate(val_ds, verbose=1)

print("\nValidation Metrics:")
print(f"  - Loss: {val_metrics[0]:.4f}")
print(f"  - Accuracy: {val_metrics[1]:.4f}")
print(f"  - Precision: {val_metrics[2]:.4f}")
print(f"  - Recall: {val_metrics[3]:.4f}")

# Calculate F1 score
f1_score = 2 * (val_metrics[2] * val_metrics[3]) / (val_metrics[2] + val_metrics[3])
print(f"  - F1 Score: {f1_score:.4f}")

### 8.1 Generate Predictions

In [ ]:
# Get predictions for validation set
print("Generating predictions...")
y_pred_probs = model.predict(val_ds, verbose=1)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

# Get true labels
y_true = np.concatenate([y for x, y in val_ds], axis=0)

print(f"\nPredictions shape: {y_pred.shape}")
print(f"True labels shape: {y_true.shape}")
print(f"Prediction probability range: [{y_pred_probs.min():.3f}, {y_pred_probs.max():.3f}]")

### 8.2 Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={'label': 'Count'}
)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Print detailed metrics
print("\nConfusion Matrix:")
print(cm)
print(f"\nTrue Negatives: {cm[0, 0]}")
print(f"False Positives: {cm[0, 1]}")
print(f"False Negatives: {cm[1, 0]}")
print(f"True Positives: {cm[1, 1]}")

### 8.3 Classification Report

In [ ]:
# Generate classification report
print("Classification Report:\n")
report = classification_report(
    y_true, 
    y_pred, 
    target_names=class_names,
    digits=4
)
print(report)

### 8.4 ROC Curve and AUC

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_true, y_pred_probs)
auc_score = roc_auc_score(y_true, y_pred_probs)

# Plot ROC curve
plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'ROC Curve (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"AUC Score: {auc_score:.4f}")

### 8.5 Precision-Recall Curve

In [ ]:
# Calculate precision-recall curve
precision, recall, pr_thresholds = precision_recall_curve(y_true, y_pred_probs)

# Plot precision-recall curve
plt.figure(figsize=(10, 6))
plt.plot(recall, precision, linewidth=2, label='Precision-Recall Curve')
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curve', fontsize=14, fontweight='bold')
plt.legend(loc='lower left', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('precision_recall_curve.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.6 Visualize Predictions

In [ ]:
def visualize_predictions(dataset, model, num_samples=10, title="Predictions"):
    """Visualize model predictions on sample images."""
    # Get a batch of images
    for images, labels in dataset.take(1):
        predictions = model.predict(images, verbose=0)
        
        # Limit to num_samples
        num_samples = min(num_samples, len(images))
        
        # Create figure
        fig, axes = plt.subplots(2, 5, figsize=(15, 6))
        fig.suptitle(title, fontsize=16, fontweight='bold')
        
        for i in range(num_samples):
            ax = axes[i // 5, i % 5]
            
            # Display image
            ax.imshow(images[i].numpy().astype("uint8"))
            
            # Get prediction
            pred_prob = predictions[i][0]
            pred_class = 1 if pred_prob > 0.5 else 0
            true_class = int(labels[i].numpy())
            
            # Set title color based on correctness
            color = 'green' if pred_class == true_class else 'red'
            
            # Title with prediction info
            title_text = f"True: {class_names[true_class]}\n"
            title_text += f"Pred: {class_names[pred_class]} ({pred_prob:.2f})"
            ax.set_title(title_text, fontsize=9, color=color, fontweight='bold')
            ax.axis('off')
        
        plt.tight_layout()
        plt.savefig('predictions_visualization.png', dpi=150, bbox_inches='tight')
        plt.show()
        break

visualize_predictions(val_ds, model, num_samples=10, title="Validation Set Predictions")

### 8.7 Find Misclassified Examples

In [ ]:
# Find indices of misclassified examples
misclassified_indices = np.where(y_pred != y_true)[0]
num_misclassified = len(misclassified_indices)

print(f"Total misclassified samples: {num_misclassified}")
print(f"Error rate: {num_misclassified / len(y_true) * 100:.2f}%")

if num_misclassified > 0:
    # Show some misclassified examples
    print(f"\nShowing {min(5, num_misclassified)} misclassified examples:")
    for idx in misclassified_indices[:5]:
        true_label = class_names[int(y_true[idx])]
        pred_label = class_names[y_pred[idx]]
        prob = y_pred_probs[idx][0]
        print(f"  Index {idx}: True={true_label}, Pred={pred_label}, Prob={prob:.3f}")

## 9. Save Model for Production

In [ ]:
# Save the trained model in Keras format
model_path = CHECKPOINT_DIR / 'final_model.keras'
model.save(model_path)
print(f"Model saved to: {model_path}")

# Save model configuration
config = {
    'model_type': 'custom_cnn',
    'input_shape': INPUT_SHAPE,
    'image_height': IMG_HEIGHT,
    'image_width': IMG_WIDTH,
    'channels': IMG_CHANNELS,
    'num_classes': 1,
    'class_names': class_names,
    'threshold': 0.5,
    'preprocessing': {
        'rescale': '1./255',
        'method': 'built-in'  # Model includes rescaling layer
    },
    'training_metrics': {
        'val_accuracy': float(val_metrics[1]),
        'val_precision': float(val_metrics[2]),
        'val_recall': float(val_metrics[3]),
        'val_f1': float(f1_score),
        'auc': float(auc_score)
    },
    'training_date': datetime.now().isoformat(),
    'total_params': model.count_params()
}

config_path = CHECKPOINT_DIR / 'model_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"Model config saved to: {config_path}")
print("\nModel Info:")
print(json.dumps(config, indent=2))

## 10. Export to TensorFlow.js

Convert the model to TensorFlow.js format for deployment in the Chrome extension.

In [ ]:
import tensorflowjs as tfjs

# Export to TensorFlow.js format
tfjs_path = str(OUTPUT_DIR)
print(f"Exporting model to TensorFlow.js format...")
print(f"Output directory: {tfjs_path}")

tfjs.converters.save_keras_model(model, tfjs_path)

print("\n✅ Model successfully exported to TensorFlow.js!")
print(f"\nFiles created in {tfjs_path}:")
for file in OUTPUT_DIR.glob('*'):
    size_kb = file.stat().st_size / 1024
    print(f"  - {file.name} ({size_kb:.1f} KB)")

In [ ]:
# Save model config to Chrome extension directory
chrome_config_path = OUTPUT_DIR / 'model_config.json'
with open(chrome_config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"Model config copied to Chrome extension: {chrome_config_path}")
print("\n✅ Model is ready for deployment in the Chrome extension!")

## 11. Test TensorFlow.js Model

In [ ]:
# Optional: Test loading the TFJS model
print("Testing TensorFlow.js model loading...")

# Load the converted model
loaded_model = tf.keras.models.load_model(model_path)

# Test prediction
for images, labels in val_ds.take(1):
    # Original model
    original_pred = model.predict(images[:1], verbose=0)
    # Loaded model
    loaded_pred = loaded_model.predict(images[:1], verbose=0)
    
    print(f"\nOriginal model prediction: {original_pred[0][0]:.6f}")
    print(f"Loaded model prediction: {loaded_pred[0][0]:.6f}")
    print(f"Difference: {abs(original_pred[0][0] - loaded_pred[0][0]):.6e}")
    
    if abs(original_pred[0][0] - loaded_pred[0][0]) < 1e-5:
        print("\n✅ Model conversion successful! Predictions match.")
    else:
        print("\n⚠️ Warning: Predictions differ slightly. This may be normal.")
    
    break

## 12. Summary and Next Steps

In [ ]:
print("=" * 80)
print("TRAINING SUMMARY")
print("=" * 80)
print(f"\nModel Architecture: Custom CNN")
print(f"Total Parameters: {model.count_params():,}")
print(f"\nDataset:")
print(f"  - Training samples: ~{train_size}")
print(f"  - Validation samples: ~{val_size}")
print(f"  - Classes: {class_names}")
print(f"\nPerformance Metrics:")
print(f"  - Validation Accuracy: {val_metrics[1]:.4f}")
print(f"  - Validation Precision: {val_metrics[2]:.4f}")
print(f"  - Validation Recall: {val_metrics[3]:.4f}")
print(f"  - F1 Score: {f1_score:.4f}")
print(f"  - AUC: {auc_score:.4f}")
print(f"\nModel Files:")
print(f"  - Keras model: {model_path}")
print(f"  - TensorFlow.js model: {OUTPUT_DIR}/model.json")
print(f"  - Config: {chrome_config_path}")
print("\n" + "=" * 80)
print("NEXT STEPS")
print("=" * 80)
print("\n1. Review training metrics above")
print("   - Target: >90% validation accuracy for production")
print("   - If accuracy is low, generate more training data")
print("\n2. Test the model in the Chrome extension")
print("   - Load extension from ../chromeExtension/")
print("   - Verify model loads correctly")
print("   - Test on real websites")
print("\n3. If performance is unsatisfactory:")
print("   - Generate more training data (1000+ images per class)")
print("   - Adjust model architecture or hyperparameters")
print("   - Try transfer learning models (MobileNet/EfficientNet)")
print("\n4. Update Chrome extension code")
print("   - Modify content.js for image capture")
print("   - Update adTargetingScript.js for CNN inference")
print("   - See conversion-todo.md Phase 3 for details")
print("\n" + "=" * 80)
print("\n🎉 Training complete! Model exported successfully.")
print("=" * 80)